In [1]:
%load_ext autoreload

%autoreload 2

In [2]:
from submit_test import ModelTester

/dss/dsshome1/00/di54xat/cloudsen12/.venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:

from src.misc import select_patches_from_dataset
from src.dataset import TestS2TIFDataSet, TestS2TIFDataSet512
from src.model import UNet
from src.training.lwf_unet_aspp_trainer import UNet as ASPPUNet
from pathlib import Path


# Testing models
Idea is to train on no-label/scribble and test on high from cloudsen12

In [18]:

def define_tester(
    user: str,
    repo: str = "cloudsen12",
    seed:int = 42,
    csv_name: str = "cloudsen12_initial_cloudfree_high.csv",
    epochs: int = 16,
    patch_size: int = 256,
    batch_size: int = 12,
    num_workers: int = 16,
    prefetch_factor: int = 8,
    experiment_id: str = "cloudsen12_aspp_scribble_multitypecloud_256_001",
    partition: str = "hpda2_compute_gpu",
    time: str = "03:00:00",
    gpus_per_node: int = 1,
    mem_gb: int = 256,
    account: str = "pn39sa-c",
    clusters: str = "hpda2",
    mail_user: str | None = None,
    exclude_nodes: str | None = None,
):

    root_hpc = Path("/dss/dsstbyfs02/pn49ci/pn49ci-dss-0026")
    user_path = root_hpc / user
    log_dir = user_path / "experiments/LWF-DLR/slurm_logs/test"
    log_dir.mkdir(parents=True, exist_ok=True)

    cpus_per_task = 2 + num_workers

    experiment_group: str = "LWF-DLR"

    data_path = user_path / repo / "data"

    # Load dataset
    file_names = select_patches_from_dataset(
        data_path / csv_name, 
        data_path,
        type_folder="", # not supported here
    )

    use_aspp_trainer = True
    if use_aspp_trainer:
        model = ASPPUNet()
    else:
        model = UNet()


    if patch_size == 512:
        dataset = TestS2TIFDataSet512(
            file_names,
            seed
        )
    else: #default
        dataset = TestS2TIFDataSet(
            file_names,
            seed
        )

    model_name = "best_model.pth"
    tester = ModelTester(
        model,
        dataset,
        batch_size,
        num_workers,
        "cuda",
        user,
        repo,
        experiment_group, 
        experiment_id,
        model_name,
    )


    return tester

In [19]:
tester = define_tester("di54xat")

In [20]:
print(tester())

Test:   0%|          | 0/167 [00:00<?, ?it/s]

Test: 100%|██████████| 167/167 [00:24<00:00,  6.72it/s]


{
    "cross_entropy": 1.7027366982248728,
    "dice_loss": 0.6690645478442757,
    "pixel_accuracy": 0.002995414733886719,
    "dice_per_class": [
        0.0009780124488534961,
        0.003147817771088011,
        0.0,
        0.004689379486704991
    ],
    "mean_dice": 0.002203802426661625,
    "iou_per_class": [
        0.0004960198695068821,
        0.0021060368984713454,
        0.0,
        0.0032139881388412667
    ],
    "mean_iou": 0.0014540112267048735
}
